# CPCB Multi-City Ambient Air Quality Benchmark System
### Cross-City Pollution Comparison, Causal Machine Learning Forecasting & Spatial Domain Generalization
**Cities Covered:** Delhi, Kolkata, Mumbai, and Bengaluru  
**Data Source:** Central Pollution Control Board (CPCB), Ministry of Environment, Forest and Climate Change, Govt. of India  
**Target:** Particulate Matter ($PM_{2.5}$) and Multi-Parameter Environmental Dynamics ($PM_{10}, SO_2, NO_2, CO, O_3, 	ext{Benzene}, 	ext{Toluene}$)  
**Evaluation Protocol:** Strict Out-of-Sample Chronological Partitioning (70% Train / 15% Validation / 15% Test) with Zero Data Leakage


In [1]:
# Section 1: Environment Setup & Modular Imports
import os
import sys
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# Ensure local src package is accessible
sys.path.insert(0, os.path.abspath('.'))

from src.multi_city import (
    load_and_prepare_multicity_data,
    get_city_clean_subset,
    plot_multicity_2x2_monthly_bars,
    plot_multicity_2x2_yearly_bars,
    plot_multicity_pollution_distributions,
    plot_multicity_aqi_severity_breakdown,
    build_city_features,
    prepare_city_splits,
    train_and_evaluate_city_models,
    plot_multicity_actual_vs_predicted_20pct,
    plot_multicity_spider_chart,
    plot_multicity_taylor_diagram,
    compute_and_plot_cross_city_transferability
)

print("Environment initialized successfully.")
print(f"Working Directory: {os.getcwd()}")


Environment initialized successfully.
Working Directory: d:\csir_rework


---
## Section 2: CPCB Multi-City Data Ingestion & Profiling
Loads standardized continuous daily observations across **Delhi, Kolkata, Mumbai, and Bengaluru**, capturing the diverse urban typologies of India:
* **Delhi**: Northern landlocked megacity subject to intense winter thermal inversions and crop residue transport.
* **Kolkata**: Eastern riverine industrial delta with humid monsoonal cycles and seasonal industrial smog.
* **Mumbai**: Western coastal metropolis with active diurnal land-sea breeze ventilation.
* **Bengaluru**: Southern high-altitude Deccan plateau with moderate temperatures and vehicular-driven emission profiles.


In [2]:
# Section 2: Multi-City Dataset Loading & Station Overview
csv_path = 'data/cpcb_multi_city_delhi_kolkata_mumbai_bengaluru.csv'
df_multi = load_and_prepare_multicity_data(csv_path)

target_cities = ['Delhi', 'Kolkata', 'Mumbai', 'Bengaluru']

print("=" * 85)
print("             CPCB MULTI-CITY CONTINUOUS AIR QUALITY DATASET AUDIT")
print("=" * 85)
print(f"Total Combined Records: {len(df_multi)} daily observations")
print(f"Temporal Coverage:      {df_multi['Date'].min().strftime('%Y-%m-%d')} to {df_multi['Date'].max().strftime('%Y-%m-%d')}")
print(f"Target Metros:          {', '.join(target_cities)}")
print("-" * 85)

for city in target_cities:
    sub = df_multi[df_multi['City'] == city]
    valid_pm = sub['PM2.5'].dropna()
    print(f"City: {city:<12} | Total Rows: {len(sub):4d} | Valid PM2.5: {len(valid_pm):4d} | Mean PM2.5: {valid_pm.mean():6.2f} ug/m3 | Max: {valid_pm.max():6.2f} ug/m3")

print("-" * 85)
display(df_multi.head())


             CPCB MULTI-CITY CONTINUOUS AIR QUALITY DATASET AUDIT
Total Combined Records: 6841 daily observations
Temporal Coverage:      2015-01-01 to 2020-07-01
Target Metros:          Delhi, Kolkata, Mumbai, Bengaluru
-------------------------------------------------------------------------------------
City: Delhi        | Total Rows: 2009 | Valid PM2.5: 2007 | Mean PM2.5: 117.20 ug/m3 | Max: 685.36 ug/m3
City: Kolkata      | Total Rows:  814 | Valid PM2.5:  759 | Mean PM2.5:  64.36 ug/m3 | Max: 304.74 ug/m3
City: Mumbai       | Total Rows: 2009 | Valid PM2.5:  784 | Mean PM2.5:  35.20 ug/m3 | Max: 134.32 ug/m3
City: Bengaluru    | Total Rows: 2009 | Valid PM2.5: 1863 | Mean PM2.5:  35.82 ug/m3 | Max: 313.59 ug/m3
-------------------------------------------------------------------------------------


,City,Date,PM2.5,PM10,NO,NO2,NOx,NH3,CO,SO2,O3,Benzene,Toluene,Xylene,AQI,AQI_Bucket,Year,Month
0,Bengaluru,2015-01-01,NaN,NaN,3.26,17.33,10.88,20.36,0.33,3.54,10.73,0.56,4.64,NaN,NaN,NaN,2015,1
1,Bengaluru,2015-01-02,NaN,NaN,6.05,19.73,14.14,23.74,1.35,3.97,22.77,0.65,5.31,NaN,NaN,NaN,2015,1
2,Bengaluru,2015-01-03,NaN,NaN,11.91,19.88,20.72,4.32,17.40,13.61,12.03,0.53,19.25,NaN,NaN,NaN,2015,1
3,Bengaluru,2015-01-04,NaN,NaN,7.45,21.61,16.88,0.87,5.05,6.52,17.70,0.55,7.47,NaN,NaN,NaN,2015,1
4,Bengaluru,2015-01-05,NaN,NaN,9.52,22.17,21.76,31.38,1.83,4.71,12.72,0.40,4.36,NaN,NaN,NaN,2015,1


---
## Section 3: Multi-City Monthly & Yearly Mean Parameter Variations (2x2 Vertical Bar Charts)
Evaluates seasonal cycles (January–December monthly means) and multi-year secular trajectories (2015–2020 yearly means) across the 4 major metros in publication-grade **2x2 vertical bar charts**.
Each bar includes exact concentration annotations and official India National Ambient Air Quality Standards (NAAQS) permissible limits.


In [3]:
# Section 3: 2x2 Vertical Bar Graphs - Monthly & Yearly Concentrations
# 1. Monthly Mean PM2.5 (Fine Particulate Matter) Across 4 Cities
fig_path_m_pm25 = 'reports/figures/multicity_monthly_pm2_5.png'
plot_multicity_2x2_monthly_bars(df_multi, target_cities, param='PM2.5', save_path=fig_path_m_pm25)
img = plt.imread(fig_path_m_pm25)
fig, ax = plt.subplots(figsize=(15, 10))
ax.imshow(img); ax.axis('off'); plt.show()

# 2. Monthly Mean PM10 (Coarse Particulate Matter) Across 4 Cities
fig_path_m_pm10 = 'reports/figures/multicity_monthly_pm10.png'
plot_multicity_2x2_monthly_bars(df_multi, target_cities, param='PM10', save_path=fig_path_m_pm10)
img = plt.imread(fig_path_m_pm10)
fig, ax = plt.subplots(figsize=(15, 10))
ax.imshow(img); ax.axis('off'); plt.show()

# 3. Monthly Mean Volatile Organic Compound: Benzene Across 4 Cities
fig_path_m_benz = 'reports/figures/multicity_monthly_benzene.png'
plot_multicity_2x2_monthly_bars(df_multi, target_cities, param='Benzene', save_path=fig_path_m_benz)
img = plt.imread(fig_path_m_benz)
fig, ax = plt.subplots(figsize=(15, 10))
ax.imshow(img); ax.axis('off'); plt.show()

# 4. Monthly Mean Gaseous Criteria Pollutant: SO2 Across 4 Cities
fig_path_m_so2 = 'reports/figures/multicity_monthly_so2.png'
plot_multicity_2x2_monthly_bars(df_multi, target_cities, param='SO2', save_path=fig_path_m_so2)
img = plt.imread(fig_path_m_so2)
fig, ax = plt.subplots(figsize=(15, 10))
ax.imshow(img); ax.axis('off'); plt.show()

# 5. Annual Secular Trend (2015-2020) PM2.5 Across 4 Cities
fig_path_y_pm25 = 'reports/figures/multicity_yearly_pm2_5.png'
plot_multicity_2x2_yearly_bars(df_multi, target_cities, param='PM2.5', save_path=fig_path_y_pm25)
img = plt.imread(fig_path_y_pm25)
fig, ax = plt.subplots(figsize=(15, 10))
ax.imshow(img); ax.axis('off'); plt.show()


C:\Users\amank\AppData\Local\Temp\ipykernel_38152\221510055.py:7: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  ax.imshow(img); ax.axis('off'); plt.show()
C:\Users\amank\AppData\Local\Temp\ipykernel_38152\221510055.py:14: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  ax.imshow(img); ax.axis('off'); plt.show()
C:\Users\amank\AppData\Local\Temp\ipykernel_38152\221510055.py:21: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  ax.imshow(img); ax.axis('off'); plt.show()
C:\Users\amank\AppData\Local\Temp\ipykernel_38152\221510055.py:28: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  ax.imshow(img); ax.axis('off'); plt.show()
C:\Users\amank\AppData\Local\Temp\ipykernel_38152\221510055.py:35: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  ax.imshow(img); ax.axis('off'); plt.show()


---
## Section 4: Cross-City Pollutant Distribution & AQI Severity Breakdown
Compares the statistical dispersion and extreme pollution episodes across the 4 metros using comparative boxplots and 100% stacked AQI category breakdowns.


In [4]:
# Section 4: Pollutant Distributions & AQI Severity Breakdown
# 1. Cross-City Pollutant Distribution Boxplots (PM2.5, PM10, NO2, SO2)
dist_fig_path = 'reports/figures/multicity_pollution_distribution_boxplots.png'
plot_multicity_pollution_distributions(df_multi, target_cities, save_path=dist_fig_path)
img = plt.imread(dist_fig_path)
fig, ax = plt.subplots(figsize=(15, 11))
ax.imshow(img); ax.axis('off'); plt.show()

# 2. Cross-City AQI Severity Category Distribution (Good to Severe)
aqi_fig_path = 'reports/figures/multicity_aqi_severity_breakdown.png'
plot_multicity_aqi_severity_breakdown(df_multi, target_cities, save_path=aqi_fig_path)
img = plt.imread(aqi_fig_path)
fig, ax = plt.subplots(figsize=(13, 6.5))
ax.imshow(img); ax.axis('off'); plt.show()


C:\Users\amank\AppData\Local\Temp\ipykernel_38152\1267923989.py:7: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  ax.imshow(img); ax.axis('off'); plt.show()
C:\Users\amank\AppData\Local\Temp\ipykernel_38152\1267923989.py:14: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  ax.imshow(img); ax.axis('off'); plt.show()


---
## Section 5: Causal Feature Engineering & Leakage-Free Partitioning (Per City)
For each city independently:
* We construct multi-horizon causal lags ($t-1, t-2, t-3, t-7, t-14$) for target and co-pollutants.
* Rolling window aggregation (mean, standard deviation, maximum, minimum over 3, 7, 14, 30 days) computed strictly on prior observations ($t-1$).
* Environmental chemical proxy ratios: Aerosol ratio ($PM_{2.5} / PM_{10}$) and stationary-mobile combustion index ($NO_2 / SO_2$).
* **Zero Leakage**: All raw calendar integers (`Year`, `Month`, `Day`) are excluded. Splits are chronological (70% Train, 15% Validation, 15% Test) with imputations fitted strictly on Training partitions.


In [5]:
# Section 5: Individual City Machine Learning Model Training & Evaluation
all_city_metrics = []
city_eval_dict = {}
city_models_dict = {}
city_data_dict = {}

for city in target_cities:
    print("=" * 80)
    print(f"TRAINING PREDICTIVE MODELS FOR: {city.upper()}")
    print("=" * 80)
    c_clean = get_city_clean_subset(df_multi, city)
    c_feat = build_city_features(c_clean, target_col='PM2.5')
    
    X_tr, y_tr, X_va, y_va, X_te, y_te, f_cols, d_te = prepare_city_splits(c_feat, target_col='PM2.5')
    print(f"Splits -> Train: {len(X_tr)} | Val: {len(X_va)} | Test: {len(X_te)} ({len(f_cols)} causal predictors)")

    city_data_dict[city] = {
        'X_train': X_tr, 'y_train': y_tr,
        'X_val': X_va, 'y_val': y_va,
        'X_test': X_te, 'y_test': y_te,
        'feature_cols': f_cols,
        'dates_test': d_te
    }

    trained_m, m_df, test_preds = train_and_evaluate_city_models(city, X_tr, y_tr, X_va, y_va, X_te, y_te)
    all_city_metrics.append(m_df)
    
    # Select winning base model and best overall predictor
    best_base = m_df[m_df['Model'] != 'Weighted Ensemble'].sort_values('Test_R2', ascending=False).iloc[0]
    best_overall = m_df.sort_values('Test_R2', ascending=False).iloc[0]
    
    city_models_dict[city] = trained_m[best_base['Model']]
    city_eval_dict[city] = {
        'y_test': y_te,
        'y_pred': test_preds[best_overall['Model']],
        'best_model_name': best_overall['Model'],
        'best_metrics': best_overall.to_dict()
    }

# Save consolidated benchmark
consolidated_df = pd.concat(all_city_metrics, ignore_index=True)
consolidated_df.to_csv('reports/multicity_model_performance_benchmark.csv', index=False)


TRAINING PREDICTIVE MODELS FOR: DELHI
Splits -> Train: 1385 | Val: 296 | Test: 298 (61 causal predictors)

[Delhi] Training candidate models...
  - Random Forest   | Val R2: 0.9610 | Test R2: 0.9423 | RMSE: 20.97 | MAE: 13.36
  - Extra Trees     | Val R2: 0.9659 | Test R2: 0.9476 | RMSE: 19.99 | MAE: 13.23
  - XGBoost         | Val R2: 0.9710 | Test R2: 0.9512 | RMSE: 19.29 | MAE: 12.92


C:\Users\amank\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.12_qbz5n2kfra8p0\LocalCache\local-packages\Python312\site-packages\sklearn\utils\validation.py:2691: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  warnings.warn(
C:\Users\amank\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.12_qbz5n2kfra8p0\LocalCache\local-packages\Python312\site-packages\sklearn\utils\validation.py:2691: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  warnings.warn(


  - LightGBM        | Val R2: 0.9658 | Test R2: 0.9361 | RMSE: 22.08 | MAE: 13.73
  - CatBoost        | Val R2: 0.9659 | Test R2: 0.9405 | RMSE: 21.31 | MAE: 14.96
  - Ridge Baseline  | Val R2: 0.9586 | Test R2: 0.9274 | RMSE: 23.53 | MAE: 14.17
  * Weighted Ensemble | Test R2: 0.9487 | RMSE: 19.79 | MAE: 13.50
TRAINING PREDICTIVE MODELS FOR: KOLKATA
Splits -> Train: 548 | Val: 117 | Test: 119 (61 causal predictors)

[Kolkata] Training candidate models...
  - Random Forest   | Val R2: 0.8216 | Test R2: 0.9553 | RMSE: 3.50 | MAE: 2.84
  - Extra Trees     | Val R2: 0.8423 | Test R2: 0.9511 | RMSE: 3.67 | MAE: 2.94
  - XGBoost         | Val R2: 0.8535 | Test R2: 0.9654 | RMSE: 3.09 | MAE: 2.43


C:\Users\amank\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.12_qbz5n2kfra8p0\LocalCache\local-packages\Python312\site-packages\sklearn\utils\validation.py:2691: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  warnings.warn(
C:\Users\amank\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.12_qbz5n2kfra8p0\LocalCache\local-packages\Python312\site-packages\sklearn\utils\validation.py:2691: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  warnings.warn(


  - LightGBM        | Val R2: 0.7614 | Test R2: 0.9669 | RMSE: 3.02 | MAE: 2.41
  - CatBoost        | Val R2: 0.7495 | Test R2: 0.9323 | RMSE: 4.31 | MAE: 3.20
  - Ridge Baseline  | Val R2: 0.8478 | Test R2: 0.9592 | RMSE: 3.35 | MAE: 2.54
  * Weighted Ensemble | Test R2: 0.9711 | RMSE: 2.82 | MAE: 2.25
TRAINING PREDICTIVE MODELS FOR: MUMBAI
Splits -> Train: 537 | Val: 115 | Test: 116 (61 causal predictors)

[Mumbai] Training candidate models...
  - Random Forest   | Val R2: 0.7077 | Test R2: 0.8521 | RMSE: 4.16 | MAE: 2.89
  - Extra Trees     | Val R2: 0.5571 | Test R2: 0.8503 | RMSE: 4.18 | MAE: 2.81
  - XGBoost         | Val R2: 0.7278 | Test R2: 0.8403 | RMSE: 4.32 | MAE: 3.12


C:\Users\amank\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.12_qbz5n2kfra8p0\LocalCache\local-packages\Python312\site-packages\sklearn\utils\validation.py:2691: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  warnings.warn(
C:\Users\amank\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.12_qbz5n2kfra8p0\LocalCache\local-packages\Python312\site-packages\sklearn\utils\validation.py:2691: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  warnings.warn(


  - LightGBM        | Val R2: 0.7120 | Test R2: 0.8609 | RMSE: 4.03 | MAE: 2.94
  - CatBoost        | Val R2: 0.5187 | Test R2: 0.8142 | RMSE: 4.66 | MAE: 3.19
  - Ridge Baseline  | Val R2: 0.8401 | Test R2: 0.8506 | RMSE: 4.18 | MAE: 3.46
  * Weighted Ensemble | Test R2: 0.8964 | RMSE: 3.48 | MAE: 2.51
TRAINING PREDICTIVE MODELS FOR: BENGALURU
Splits -> Train: 1330 | Val: 285 | Test: 286 (61 causal predictors)

[Bengaluru] Training candidate models...
  - Random Forest   | Val R2: 0.4317 | Test R2: 0.4306 | RMSE: 10.27 | MAE: 6.53
  - Extra Trees     | Val R2: 0.8008 | Test R2: 0.6620 | RMSE: 7.92 | MAE: 6.07
  - XGBoost         | Val R2: 0.7408 | Test R2: 0.6638 | RMSE: 7.90 | MAE: 5.89


C:\Users\amank\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.12_qbz5n2kfra8p0\LocalCache\local-packages\Python312\site-packages\sklearn\utils\validation.py:2691: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  warnings.warn(
C:\Users\amank\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.12_qbz5n2kfra8p0\LocalCache\local-packages\Python312\site-packages\sklearn\utils\validation.py:2691: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  warnings.warn(


  - LightGBM        | Val R2: 0.7843 | Test R2: 0.6158 | RMSE: 8.44 | MAE: 6.17
  - CatBoost        | Val R2: 0.7847 | Test R2: 0.6416 | RMSE: 8.15 | MAE: 6.10
  - Ridge Baseline  | Val R2: 0.8394 | Test R2: 0.6982 | RMSE: 7.48 | MAE: 5.40
  * Weighted Ensemble | Test R2: 0.6892 | RMSE: 7.59 | MAE: 5.70


---
## Section 6: Consolidated Multi-City Model Performance Benchmark Matrix
Synthesizes the predictive performance of all evaluated models across Delhi, Kolkata, Mumbai, and Bengaluru on the held-out test split.


In [6]:
# Section 6: Multi-City Performance Matrix Display
benchmark_summary = consolidated_df[['City', 'Model', 'Val_R2', 'Test_R2', 'Test_RMSE', 'Test_MAE', 'Test_sMAPE_%', 'Directional_Accuracy_%', 'FAC2_%']]
display(benchmark_summary.sort_values(['City', 'Test_R2'], ascending=[True, False]))


,City,Model,Val_R2,Test_R2,Test_RMSE,Test_MAE,Test_sMAPE_%,Directional_Accuracy_%,FAC2_%
26,Bengaluru,Ridge Baseline,0.839404,0.698229,7.480032,5.403549,17.273913,54.736842,99.300699
27,Bengaluru,Weighted Ensemble,0.808299,0.689197,7.591140,5.700201,18.711036,50.526316,99.300699
23,Bengaluru,XGBoost,0.740840,0.663816,7.895021,5.887816,19.130577,53.333333,98.601399
22,Bengaluru,Extra Trees,0.800761,0.662004,7.916267,6.069437,19.981984,50.526316,98.251748
25,Bengaluru,CatBoost,0.784732,0.641558,8.152187,6.101673,20.094012,51.929825,98.601399
24,Bengaluru,LightGBM,0.784305,0.615826,8.439727,6.172713,19.760780,56.491228,98.251748
21,Bengaluru,Random Forest,0.431735,0.430632,10.274510,6.532864,20.097591,49.473684,96.853147
2,Delhi,XGBoost,0.970982,0.951218,19.287706,12.921510,13.719434,81.481481,100.000000
6,Delhi,Weighted Ensemble,0.967597,0.948661,19.786860,13.496105,14.519777,82.491582,100.000000
1,Delhi,Extra Trees,0.965876,0.947619,19.986577,13.233803,14.238542,80.471380,99.328859


---
## Section 7: Cross-City Karl Taylor Diagram Analysis
Compares the spatial skill of the winning model for each city on a single Karl Taylor polar diagram.
The plot simultaneously depicts the Pearson correlation coefficient ($r$), normalized standard deviation ($\sigma_n$), and centered root-mean-square difference ($E'$).


In [7]:
# Section 7: Multi-City Karl Taylor Polar Diagram
taylor_path = 'reports/figures/multicity_taylor_diagram.png'
plot_multicity_taylor_diagram(city_eval_dict, save_path=taylor_path)
img = plt.imread(taylor_path)
fig, ax = plt.subplots(figsize=(10, 10))
ax.imshow(img); ax.axis('off'); plt.show()


C:\Users\amank\AppData\Local\Temp\ipykernel_38152\2082529890.py:6: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  ax.imshow(img); ax.axis('off'); plt.show()


---
## Section 8: Cross-City Radar / Spider Plot Benchmark
Multi-criteria polar evaluation assessing predictive skill ($R^2$), precision, directional accuracy, and factor-of-2 compliance ($FAC2$) across all 4 cities.


In [8]:
# Section 8: Multi-Criteria Radar / Spider Chart
spider_path = 'reports/figures/multicity_spider_chart.png'
city_best_metrics = {c: city_eval_dict[c]['best_metrics'] for c in target_cities}
plot_multicity_spider_chart(city_best_metrics, save_path=spider_path)
img = plt.imread(spider_path)
fig, ax = plt.subplots(figsize=(10, 10))
ax.imshow(img); ax.axis('off'); plt.show()


C:\Users\amank\AppData\Local\Temp\ipykernel_38152\709040787.py:7: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  ax.imshow(img); ax.axis('off'); plt.show()


---
## Section 9: Actual vs. Predicted with $\pm 20\%$ Precision Error Lines (2x2 Grid)
Visualizes model reliability across Delhi, Kolkata, Mumbai, and Bengaluru on a 2x2 grid. Each panel features the $1:1$ identity line and $\pm 20\%$ precision bounds.


In [9]:
# Section 9: 2x2 Actual vs Predicted with +-20% Error Lines
act_pred_path = 'reports/figures/multicity_actual_vs_predicted_20pct.png'
plot_multicity_actual_vs_predicted_20pct(city_eval_dict, save_path=act_pred_path)
img = plt.imread(act_pred_path)
fig, ax = plt.subplots(figsize=(15, 13))
ax.imshow(img); ax.axis('off'); plt.show()


C:\Users\amank\AppData\Local\Temp\ipykernel_38152\2939963213.py:6: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  ax.imshow(img); ax.axis('off'); plt.show()


---
## Section 10: Spatial Domain Generalization & Cross-City Transferability Matrix
A major contribution of this research: evaluating whether a model trained in one city can forecast air quality in a completely unseen city.
* Models trained on Delhi, Kolkata, Mumbai, and Bengaluru are evaluated against the held-out test observations of every other city without re-training or adaptation.


In [10]:
# Section 10: Cross-City Generalization & Transferability Matrix
transfer_path = 'reports/figures/multicity_transferability_matrix.png'
transfer_df = compute_and_plot_cross_city_transferability(city_models_dict, city_data_dict, save_path=transfer_path)

print("=" * 80)
print("CROSS-CITY GENERALIZATION MATRIX (TEST R2 SCORE)")
print("=" * 80)
display(transfer_df)

img = plt.imread(transfer_path)
fig, ax = plt.subplots(figsize=(9, 7.5))
ax.imshow(img); ax.axis('off'); plt.show()


C:\Users\amank\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.12_qbz5n2kfra8p0\LocalCache\local-packages\Python312\site-packages\sklearn\utils\validation.py:2691: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  warnings.warn(
C:\Users\amank\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.12_qbz5n2kfra8p0\LocalCache\local-packages\Python312\site-packages\sklearn\utils\validation.py:2691: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  warnings.warn(
C:\Users\amank\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.12_qbz5n2kfra8p0\LocalCache\local-packages\Python312\site-packages\sklearn\utils\validation.py:2691: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  warnings.warn(
C:\Users\amank\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.12_qbz5n2kfra8p0\LocalCache\local-packages\Python312\site-packag

CROSS-CITY GENERALIZATION MATRIX (TEST R2 SCORE)


,Test: Delhi,Test: Kolkata,Test: Mumbai,Test: Bengaluru
Train: Delhi,0.951218,0.053549,-2.120871,0.429160
Train: Kolkata,0.865676,0.966891,0.059387,0.816952
Train: Mumbai,0.146115,0.746025,0.860928,0.649918
Train: Bengaluru,0.801749,0.862657,0.752040,0.698229


C:\Users\amank\AppData\Local\Temp\ipykernel_38152\295886910.py:12: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  ax.imshow(img); ax.axis('off'); plt.show()
